# 01 — Preparing market data and features

Start with daily S&P 500 prices and build a dataset for predicting the **next trading session's close**. The inputs use information available by the end of the current session: earlier closes, returns, moving averages, and other technical indicators.

This notebook deliberately keeps macroeconomic series out of the prediction dataset. For historical forecasting, their public release dates matter as much as the observation dates. The code here is part of the 2026 follow-up, not the original 2024 experiment.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Download a sample if no local CSV is available (internet required).
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
from sp500_research.features import model_features
frame = build_supervised_frame(prices)
print('Rows:',len(frame))
print('Features:',model_features(frame))
print(frame[['close_t','lag_7','ema_90','target_next_close','target_date']].head(3).to_string())
assert (frame.target_date > frame.index.to_series()).all()

In [ ]:
parts = chronological_split(frame)
for name,part in (('Train',parts.train),('Validation',parts.valid),('Test',parts.test)):
    print(f'{name}: {len(part):4d} targets from {part.target_date.min().date()} to {part.target_date.max().date()}')

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(prices.index,prices.Close,label='S&P 500 close',linewidth=1)
ax.set(title='Historical index close (descriptive)',xlabel='Date',ylabel='Index points')
ax.legend();plt.tight_layout();plt.show()

### What to look for

Check the first few rows: `lag_7` refers to a close seven sessions ago, while `target_next_close` is the following session's close. The next notebook compares these features against the simple last-close benchmark.